# Four Sigma Hackathon

## The Task

You have historical stock data for 1,000 anonymized companies: prices and trading volume at 5-minute, 30-minute, and daily intervals.

Every 30 minutes during the trading day, your model will predict: **which stocks will go up the most, and which will go down the most, over the next 30 minutes.**

You are not trying to predict exact prices. You are trying to get the **ranking** right. If you say stock A will outperform stock B, and it does, you score well. Your final score is the average [Spearman rank correlation](https://en.wikipedia.org/wiki/Spearman%27s_rank_correlation_coefficient) between your predicted rankings and the actual rankings across the entire test year.

## How This Notebook Works

1. **Load Data** — verify your dataset is correct
2. **Explore** — understand what the data looks like
3. **Build Features** — engineer the X variables that your model will learn from
4. **Evaluate Locally** — test your features on training data before submitting
5. **Package for Submission** — wrap your work into the required format
6. **Validate** — run checks to confirm your submission is correctly formatted

You can use **any Python library** for data analysis and feature engineering — scikit-learn, XGBoost, PyTorch, Prophet, statsmodels, ta-lib, or anything else. Add it to `submission/requirements.txt` and use it freely.

You **must** use ML in the final deliverables. 

## Setup

Download the training data zip from Google Drive and extract it into the `data/` folder so your directory looks like:

```
four-sigma-hackathon/
    feature-engineering-scaffold.ipynb   (this notebook)
    universe.json
    submission/
        model.py
        config.json
        requirements.txt
    data/
        train/
            {ticker}_1day.parquet
            {ticker}_30min.parquet
            {ticker}_5min.parquet
            ...
```

## Where's the Data? 

We've filtered for the 1,000 highest volume stocks, to give you the most signal for model training. 

The data download is located here: https://drive.google.com/drive/folders/1D8u8wKSJocOOdBFElq5pbImmmpPHQQw2?usp=sharing 

Please reach out to us if there's anything we can do to make your EDA phase more insightful. 

---
## 1. Load Data

In [ ]:
import json
import pathlib
import pickle
from datetime import time as dt_time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# paths — relative to this notebook
DATA_DIR = pathlib.Path("data/train")
UNIVERSE_PATH = pathlib.Path("universe.json")

assert DATA_DIR.exists(), f"data not found at {DATA_DIR} — extract the training data zip into data/"
assert UNIVERSE_PATH.exists(), f"universe.json not found — make sure it's in the same folder as this notebook"

# load universe ticker list
with open(UNIVERSE_PATH) as f:
    universe_data = json.load(f)
TICKERS = universe_data["tickers"]

# verify files
FREQUENCIES = ["1day", "30min", "5min"]
for freq in FREQUENCIES:
    count = len(list(DATA_DIR.glob(f"*_{freq}.parquet")))
    print(f"{freq}: {count} files")
    assert count == 1000, f"expected 1000 {freq} files, found {count}"

print(f"\ndata loaded: {len(TICKERS)} tickers, 3 frequencies")

# constants
REGULAR_OPEN = dt_time(9, 30)
REGULAR_CLOSE = dt_time(16, 0)

---
## 2. Explore the Data

In [ ]:
# --- data summary: schema, date ranges, basic statistics ---

# column schema per frequency
print("=" * 60)
print("COLUMN SCHEMA BY FREQUENCY")
print("=" * 60)
for freq in FREQUENCIES:
    sample_file = list(DATA_DIR.glob(f"*_{freq}.parquet"))[0]
    df = pd.read_parquet(sample_file)
    print(f"\n{freq}:")
    print(f"  columns: {list(df.columns)}")
    print(f"  dtypes:")
    for col in df.columns:
        print(f"    {col:>12s} -> {df[col].dtype}")

# date ranges and row counts across all tickers
print("\n" + "=" * 60)
print("DATE RANGES AND ROW COUNTS (sampled across 50 tickers)")
print("=" * 60)

sample_tickers = TICKERS[::20]  # every 20th ticker = 50 samples
summary_rows = []

for ticker in sample_tickers:
    for freq in FREQUENCIES:
        df = pd.read_parquet(DATA_DIR / f"{ticker}_{freq}.parquet")
        summary_rows.append({
            "ticker": ticker,
            "freq": freq,
            "rows": len(df),
            "min_date": df["datetime"].min(),
            "max_date": df["datetime"].max(),
            "min_close": df["close"].min(),
            "max_close": df["close"].max(),
            "mean_volume": df["volume"].mean(),
        })

summary = pd.DataFrame(summary_rows)

for freq in FREQUENCIES:
    sub = summary[summary["freq"] == freq]
    print(f"\n{freq}:")
    print(f"  date range:  {sub['min_date'].min().date()} to {sub['max_date'].max().date()}")
    print(f"  rows/ticker: {sub['rows'].min():,} — {sub['rows'].max():,} (median {sub['rows'].median():,.0f})")
    print(f"  close range: ${sub['min_close'].min():.4f} — ${sub['max_close'].max():.2f}")
    print(f"  avg volume:  {sub['mean_volume'].min():,.0f} — {sub['mean_volume'].max():,.0f}")

# full describe() on one representative ticker per frequency
print("\n" + "=" * 60)
print(f"DETAILED DESCRIBE — ticker {TICKERS[0]}")
print("=" * 60)
for freq in FREQUENCIES:
    df = pd.read_parquet(DATA_DIR / f"{TICKERS[0]}_{freq}.parquet")
    print(f"\n--- {freq} ({len(df):,} rows) ---")
    print(df.describe().to_string())

In [ ]:
# --- visualizations: distributions and coverage ---

# 1. row count distribution per frequency
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, freq in zip(axes, FREQUENCIES):
    counts = []
    for ticker in TICKERS:
        path = DATA_DIR / f"{ticker}_{freq}.parquet"
        counts.append(pd.read_parquet(path, columns=["datetime"]).shape[0])
    ax.hist(counts, bins=40, color="steelblue", edgecolor="none", alpha=0.8)
    ax.set_title(f"{freq} — rows per ticker")
    ax.set_xlabel("row count")
    ax.set_ylabel("number of tickers")
    ax.axvline(np.median(counts), color="coral", linestyle="--", label=f"median: {np.median(counts):,.0f}")
    ax.legend()
plt.suptitle("row count distributions across 1,000 tickers", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

# 2. price and volume distributions (30min, sampled tickers)
close_vals = []
volume_vals = []
for ticker in sample_tickers:
    df = pd.read_parquet(DATA_DIR / f"{ticker}_30min.parquet")
    close_vals.append(df["close"].median())
    volume_vals.append(df["volume"].mean())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(close_vals, bins=40, color="teal", edgecolor="none", alpha=0.8)
axes[0].set_title("median close price per ticker (30min)")
axes[0].set_xlabel("price")
axes[0].set_ylabel("count")

axes[1].hist(np.log10(np.array(volume_vals) + 1), bins=40, color="darkorange", edgecolor="none", alpha=0.8)
axes[1].set_title("mean volume per ticker — log10 scale (30min)")
axes[1].set_xlabel("log10(volume)")
axes[1].set_ylabel("count")

plt.tight_layout()
plt.show()

# 3. date coverage: earliest start date per ticker (1day)
start_dates = []
for ticker in TICKERS:
    df = pd.read_parquet(DATA_DIR / f"{ticker}_1day.parquet", columns=["datetime"])
    start_dates.append(df["datetime"].min())

start_series = pd.Series(start_dates)
fig, ax = plt.subplots(figsize=(14, 4))
ax.hist(start_series.dt.year + start_series.dt.month / 12, bins=50, color="steelblue", edgecolor="none", alpha=0.8)
ax.set_title("distribution of earliest data start date (1day files)")
ax.set_xlabel("year")
ax.set_ylabel("number of tickers")
plt.tight_layout()
plt.show()

print(f"earliest start: {start_series.min().date()}")
print(f"latest start:   {start_series.max().date()}")
print(f"all end at:      2025-02-26 (train cutoff)")

# 4. 30min bars per day — check for completeness
df_check = pd.read_parquet(DATA_DIR / f"{TICKERS[0]}_30min.parquet")
df_check = df_check[(df_check["datetime"].dt.time >= REGULAR_OPEN) & (df_check["datetime"].dt.time <= REGULAR_CLOSE)]
bars_per_day = df_check.groupby(df_check["datetime"].dt.date).size()

print(f"most common bars/day: {bars_per_day.mode().values[0]}")
print(f"regular hours 9:30-16:00 = 13 half-hour bars")

Each stock has OHLCV data (open, high, low, close, volume) at three frequencies. The ticker names are anonymized hashes — the same hash always refers to the same company across all frequencies.

In [ ]:
# pick a stock and look at its data across all three frequencies
sample_ticker = TICKERS[250]
print(f"ticker: {sample_ticker}")

for freq in FREQUENCIES:
    df = pd.read_parquet(DATA_DIR / f"{sample_ticker}_{freq}.parquet")
    print(f"\n{freq}: {df.shape[0]:,} rows, {df['datetime'].min().date()} to {df['datetime'].max().date()}")
    print(df.head(3).to_string(index=False))

In [ ]:
# visualize one stock's recent price history and 30-min returns
df_30 = pd.read_parquet(DATA_DIR / f"{sample_ticker}_30min.parquet")
df_30 = df_30[(df_30["datetime"].dt.time >= REGULAR_OPEN) & (df_30["datetime"].dt.time <= REGULAR_CLOSE)]
df_30 = df_30.sort_values("datetime")

recent = df_30[df_30["datetime"] >= df_30["datetime"].max() - pd.Timedelta(days=14)]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(recent["datetime"], recent["close"], color="steelblue", linewidth=1)
axes[0].set_ylabel("close price")
axes[0].set_title(f"stock {sample_ticker} — last 2 weeks of 30-min bars")

recent_return = recent["close"].pct_change()
colors = ["green" if r > 0 else "red" for r in recent_return]
axes[1].bar(recent["datetime"], recent_return, color=colors, width=0.01, alpha=0.7)
axes[1].set_ylabel("30-min return")
axes[1].set_xlabel("datetime")
axes[1].axhline(0, color="black", linewidth=0.5)

plt.tight_layout()
plt.show()

print("each bar is one 30-minute return. your model predicts these — which stocks go up, which go down.")

---
## 3. Build Features

Features are the X variables your model learns from. The raw data gives you price and volume. Your job is to transform those into signals that help predict the next 30 minutes.

**Important rule:** features at time t can only use data up to time t. No peeking at future data. We will verify this.

Below are examples at three levels. Use these as starting points, combine them, or build your own.

### Starter: Momentum and Volume Change

The simplest features — how much did the stock move in the last bar, and how much did volume change?

In [ ]:
# compute starter features for all tickers from 30min data
# builds a dict: timestamp -> {ticker: {feature_name: value}}

features_by_timestamp = {}

for i, ticker in enumerate(TICKERS):
    df = pd.read_parquet(DATA_DIR / f"{ticker}_30min.parquet")
    df = df[(df["datetime"].dt.time >= REGULAR_OPEN) & (df["datetime"].dt.time <= REGULAR_CLOSE)]
    df = df.sort_values("datetime")

    for _, day in df.groupby(df["datetime"].dt.date):
        day = day.sort_values("datetime")
        momentum = day["close"].pct_change()
        vol_change = day["volume"] / day["volume"].shift(1)
        bar_range = (day["high"] - day["low"]) / day["close"]

        for ts, mom, vc, br in zip(day["datetime"], momentum, vol_change, bar_range):
            if np.isnan(mom):
                continue
            if ts not in features_by_timestamp:
                features_by_timestamp[ts] = {}
            features_by_timestamp[ts][ticker] = {
                "momentum": mom,
                "volume_change": vc if not np.isnan(vc) and not np.isinf(vc) else 0.0,
                "bar_range": br,
            }

    if (i + 1) % 200 == 0:
        print(f"  processed {i + 1} / {len(TICKERS)} tickers")

print(f"\nfeatures computed for {len(features_by_timestamp)} timestamps")

In [ ]:
# save features so you can skip the computation on kernel restart
with open("features_cache.pkl", "wb") as f:
    pickle.dump(features_by_timestamp, f)
print(f"saved to features_cache.pkl")

# to reload after kernel restart, uncomment and run this instead of the cell above:
# with open("features_cache.pkl", "rb") as f:
#     features_by_timestamp = pickle.load(f)
# print(f"loaded {len(features_by_timestamp)} timestamps from cache")

### Intermediate: Rolling Statistics

Compute features over a lookback window — rolling mean, rolling volatility. These capture trends and regimes that single-bar features miss.

In [ ]:
# example: rolling features for a single ticker
# loop over all tickers and add these to features_by_timestamp for your full pipeline

sample = pd.read_parquet(DATA_DIR / f"{TICKERS[0]}_30min.parquet")
sample = sample[(sample["datetime"].dt.time >= REGULAR_OPEN) & (sample["datetime"].dt.time <= REGULAR_CLOSE)]
sample = sample.sort_values("datetime")
sample["return"] = sample["close"].pct_change()

sample["rolling_mean_5"] = sample["return"].rolling(5).mean()
sample["rolling_std_5"] = sample["return"].rolling(5).std()
sample["rolling_volume_ratio"] = sample["volume"] / sample["volume"].rolling(5).mean()
sample["rolling_mean_13"] = sample["return"].rolling(13).mean()
sample["rolling_std_13"] = sample["return"].rolling(13).std()

print("rolling features for one ticker:")
print(sample[["datetime", "return", "rolling_mean_5", "rolling_std_5", "rolling_volume_ratio"]].dropna().tail(8).to_string(index=False))
print()
print("longer windows capture slower trends. shorter windows capture recent momentum.")

### Advanced: Cross-Sectional Features and Multi-Timeframe

Compare each stock to the rest of the universe at each timestamp. Or use 5-minute data for finer-grained signals and aggregate up.

Ideas to explore:
- **Cross-sectional z-score:** how far is this stock's return from the universe mean, in standard deviations?
- **Sector-relative momentum:** cluster stocks by correlation, compare within clusters
- **5-min microstructure:** compute features from 5-min bars (intraday volatility, volume spikes) and attach to the 30-min grid
- **Time-of-day effects:** some times are systematically different — opening, closing, lunch
- **External libraries:** Prophet for trend decomposition, ta-lib for technical indicators, any ML library for learned features

In [ ]:
# example: cross-sectional z-score at one timestamp
sample_ts = sorted(features_by_timestamp.keys())[5000]
cross_section = features_by_timestamp[sample_ts]

mom_values = {t: cross_section[t]["momentum"] for t in cross_section}
mom_series = pd.Series(mom_values)

cs_mean = mom_series.mean()
cs_std = mom_series.std()
z_scores = (mom_series - cs_mean) / cs_std

print(f"timestamp: {sample_ts}")
print(f"stocks: {len(z_scores)}")
print(f"momentum range: {mom_series.min():.4f} to {mom_series.max():.4f}")
print(f"z-score range: {z_scores.min():.2f} to {z_scores.max():.2f}")
print(f"\ntop 5 (strongest relative to peers):")
print(z_scores.sort_values(ascending=False).head().to_string())
print(f"\nbottom 5:")
print(z_scores.sort_values().head().to_string())

---
## 4. Evaluate Locally

Before you submit, test whether your features have signal. Split your training data into a train and validation period, generate predictions on the validation period, and compute Spearman IC — the same metric we use for scoring.

A consistently positive IC on validation data means your features are finding real patterns.

In [ ]:
from scipy.stats import spearmanr

# use the last 3 months of training data as a validation set
all_ts = sorted(features_by_timestamp.keys())
val_cutoff = all_ts[-1] - pd.Timedelta(days=90)
val_timestamps = [ts for ts in all_ts if ts >= val_cutoff]
print(f"validation period: {val_timestamps[0]} to {val_timestamps[-1]}")
print(f"validation timestamps: {len(val_timestamps)}")

ic_values = []
for ts in val_timestamps:
    if ts not in features_by_timestamp:
        continue
    cross_section = features_by_timestamp[ts]
    # predicted score = momentum (replace with your model's output)
    predictions = {t: cross_section[t]["momentum"] for t in cross_section}
    # find the next timestamp for the realized return
    ts_idx = all_ts.index(ts)
    if ts_idx + 1 >= len(all_ts):
        continue
    next_ts = all_ts[ts_idx + 1]
    if next_ts.date() != ts.date():
        continue
    if next_ts not in features_by_timestamp:
        continue
    realized = {t: features_by_timestamp[next_ts][t]["momentum"]
                for t in features_by_timestamp[next_ts]
                if t in predictions}
    common = set(predictions.keys()) & set(realized.keys())
    if len(common) < 100:
        continue
    pred_arr = np.array([predictions[t] for t in common])
    real_arr = np.array([realized[t] for t in common])
    ic, _ = spearmanr(pred_arr, real_arr)
    if not np.isnan(ic):
        ic_values.append(ic)

ic_array = np.array(ic_values)
print(f"\nlocal validation ({len(ic_values)} timestamps):")
print(f"  mean IC:  {ic_array.mean():.4f}")
print(f"  std IC:   {ic_array.std():.4f}")
print(f"  pct > 0:  {(ic_array > 0).mean():.1%}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(ic_array, bins=50, color="steelblue", edgecolor="none", alpha=0.8)
ax.axvline(ic_array.mean(), color="coral", linestyle="--", label=f"mean: {ic_array.mean():.4f}")
ax.axvline(0, color="black", linewidth=0.5, linestyle="--")
ax.set_xlabel("Spearman IC")
ax.set_ylabel("frequency")
ax.set_title("local validation: IC distribution")
ax.legend()
plt.tight_layout()
plt.show()

print("\nmean IC near zero = no predictive power. try more features or a different approach.")

---
## 5. Package for Submission

Your submission directory should contain a `model.py` that defines a `Model` class with three methods:

- **`load(weights_path, config)`** — load your trained model weights
- **`prepare(data_dir)`** — read OHLCV data and compute your features in batch. Runs **once** on the full dataset.
- **`predict(timestamp)`** — return a score for each stock. Uses pre-computed features from `prepare()`. Must be fast.

The `submission/model.py` in this package already has a working momentum baseline. Open it, study the pattern, then replace the feature engineering with your own approach.

In [ ]:
# test your submission model locally
import importlib.util

spec = importlib.util.spec_from_file_location("sub_model", "submission/model.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

model = module.Model()
model.load("", {})
model.prepare(str(DATA_DIR))

# test predict on a sample timestamp
sample_ts = sorted(features_by_timestamp.keys())[1000]
pred = model.predict(sample_ts)
print(f"predict({sample_ts}): {len(pred)} tickers")
print(f"sample scores: {pred.head().to_string()}")

---
## 6. Validate Your Submission

Run this cell to check that your model's output is correctly formatted. This uses the same checks we run internally when scoring. All checks must pass before you submit.

In [ ]:
# self-contained validation — works after kernel restart if you reload the cache
# and re-run your submission model above

with open("universe.json") as f:
    expected_tickers = json.load(f)["tickers"]

# pick 5 timestamps spread across the scored range
all_ts = sorted(features_by_timestamp.keys())
sample_indices = np.linspace(0, len(all_ts) - 1, 5, dtype=int)

print(f"universe: {len(expected_tickers)} tickers")
print(f"your timestamps: {len(all_ts)}")
print()

all_passed = True
for i in sample_indices:
    ts = all_ts[i]
    pred = model.predict(ts)
    if not isinstance(pred, pd.Series):
        print(f"[FAIL] {ts}: predict() returned {type(pred).__name__}, expected pd.Series")
        all_passed = False
        continue

    valid = set(pred.index) & set(expected_tickers)
    issues = []
    if len(valid) == 0:
        issues.append("zero valid tickers")
    if len(pred) > 0 and not np.issubdtype(pred.dtype, np.number):
        issues.append(f"non-numeric ({pred.dtype})")
    if pred.isna().any():
        issues.append(f"{pred.isna().sum()} NaN")
    if len(pred) > 0 and np.isinf(pred.values).any():
        issues.append(f"{np.isinf(pred.values).sum()} inf")

    if issues:
        print(f"[FAIL] {ts}: {', '.join(issues)}")
        all_passed = False
    else:
        print(f"[PASS] {ts}: {len(valid)} tickers, clean")

print()
if all_passed:
    print("all checks passed — ready to submit")
else:
    print("some checks failed — fix issues above before submitting")

---
### Submission Checklist

- [ ] `submission/model.py` defines `Model` with `load()`, `prepare()`, `predict()`
- [ ] `submission/config.json` exists
- [ ] `submission/requirements.txt` lists all your dependencies
- [ ] `submission/weights.pkl` if your model has trained weights
- [ ] `submission/data/` contains your pre-computed feature data from training
- [ ] All validation checks above passed
- [ ] Your local IC is consistently positive on validation data